# PYSPARK

###  CUSTOMERS

In [0]:
#By default all the colums are string type , but here we use infer schema
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema", True)\
        .load("/Volumes/databricksharis/bronze/bronze_volume/customers/dim_customers.csv")

In [0]:
display(df)

In [0]:
df.printSchema()

### We have used uper case

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
df = df.withColumn("name",upper(df.name))

In [0]:
display(df)

## Find Domains from email

In [0]:
df = df.withColumn("domain",split(col("email"),"@")[1])
display(df)


## Aggregation on domains
* Add visualization and filters and even can you downlaod

In [0]:
display(df.groupBy("domain").\
    agg(count(col("customer_id")).alias("total_customer")).\
        sort(col("total_customer").desc()))



Databricks visualization. Run in Databricks to view.

In [0]:
df = df.withColumn("processed_date",current_timestamp())
display(df)

Different Modes 
* Append 
* Error 
* overwrite 
* ignore 
* Upsert (Update + Insert) We should have target table ready

In [0]:

df.write.format('delta')\
    .mode()

# Upsert

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("databricksharis.silver.customers_enr"):
    # We create the DeltaTable object to perform merge (upsert) operations on the existing Delta table
    dlt_obj = DeltaTable.forName(spark,"databricksharis.silver.customers_enr")
    dlt_obj.alias("trg").merge(df.alias("src"),"trg.customer_id = src.customer_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else :
    df.write.format("delta")\
        .mode("append")\
        .saveAsTable("databricksharis.silver.customers_enr")

In [0]:
%sql
---- IT'LL NOT CREATE DUPLICATES
select * from databricksharis.silver.customers_enr

### Products

In [0]:
df_prod = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema", True)\
        .load("/Volumes/databricksharis/bronze/bronze_volume/products/dim_products.csv")
display(df_prod)

In [0]:
df_prod = df_prod.withColumn("processed_date",current_timestamp())
display(df_prod)


### Add Aggregation

In [0]:
display(df_prod.groupBy(col("category")).agg(avg(col("price")).alias("avg_price")).sort(col("avg_price").desc()))

Databricks visualization. Run in Databricks to view.

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("databricksharis.silver.products_enr"):
    # We create the DeltaTable object to perform merge (upsert) operations on the existing Delta table
    dlt_obj = DeltaTable.forName(spark,"databricksharis.silver.products_enr")
    dlt_obj.alias("trg").merge(df.alias("src"),"trg.product_id = src.product_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else :
    df_prod.write.format("delta")\
        .mode("append")\
        .saveAsTable("databricksharis.silver.products_enr")

In [0]:
%sql
select * from databricksharis.silver.products_enr

### **STORES**

In [0]:
df_str = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema", True)\
        .load("/Volumes/databricksharis/bronze/bronze_volume/stores/dim_stores.csv")
display(df_str)

In [0]:
df_str = df_str.withColumn("store_name", regexp_replace(col("store_name"),"_",""))
display(df_str)

In [0]:
df_str = df_str.withColumn("processed_date", current_timestamp())
display(df_str)

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("databricksharis.silver.stores_enr"):
    # We create the DeltaTable object to perform merge (upsert) operations on the existing Delta table
    dlt_obj = DeltaTable.forName(spark,"databricksharis.silver.stores_enr")
    dlt_obj.alias("trg").merge(df.alias("src"),"trg.store_id = src.store_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else :
    df_str.write.format("delta")\
        .mode("append")\
        .saveAsTable("databricksharis.silver.stores_enr")

### **SALES**

In [0]:
df_sales = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema", True)\
        .load("/Volumes/databricksharis/bronze/bronze_volume/sales/")
display(df_sales)

In [0]:
df_sales = df_sales.withColumn("pricePerSale", round(col("total_amount")/col("quantity"),2))
display(df_sales)
df_sales = df_sales.withColumn("processed_date", current_timestamp())
display(df_sales)

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("databricksharis.silver.sales_enr"):
    # We create the DeltaTable object to perform merge (upsert) operations on the existing Delta table
    dlt_obj = DeltaTable.forName(spark,"databricksharis.silver.sales_enr")
    dlt_obj.alias("trg").merge(df.alias("src"),"trg.sales_id = src.sales_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else :
    df_sales.write.format("delta")\
        .mode("append")\
        .saveAsTable("databricksharis.silver.sales_enr")

In [0]:
%sql
select * from databricksharis.silver.sales_enr

###  **SPARK SQL**

In [0]:
df_sales_sql = spark.sql("SELECT * FROM databricksharis.silver.sales_enr")
display(df_sales_sql)

In [0]:
df = spark.sql("SELECT UPPER(category),CASE WHEN category = 'Toys' THEN 'YES' ELSE 'NO' END AS Flag FROM databricksharis.silver.products_enr")
display(df)

### **Real Case Scenario**
Convert data frame into sql objects so that we leverage sql language 

In [0]:
df_prod.createOrReplaceTempView("temp_products")

In [0]:
df= spark.sql("""select *, CASE WHEN category = "Toys" THEN 'YES'
          ELSE 'NO' END AS Flag from temp_products""")

display(df)

### **PYSPARK UDFS**
### By default it doesn't have Python interpreter , so it has to attach python

In [0]:
def greet(p_input):
    return print("Hello " + str(p_input))

In [0]:
#registers the UDF
udf_greet = udf(greet, StringType())

In [0]:
df = df.withColumn("greet", udf_greet(col("Flag")))
display(df)